# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [1]:
import os  # Para manipular rutas de archivos
from sklearn.feature_extraction.text import TfidfVectorizer  # Para calcular TF-IDF
from sklearn.metrics.pairwise import cosine_similarity  # Para calcular similitud entre vectores
import numpy as np  # Para operaciones numéricas

with open("../01_corpus_turismo_500.txt", 'r', encoding='utf-8') as f:  # Abre archivo de turismo
    textos_turismo = [linea.strip() for linea in f if linea.strip()]  # Lee cada línea y las guarda en una lista

# Crea un objeto vectorizador TF-IDF
vectorizer_turismo = TfidfVectorizer()

# Entrena el vectorizador y convierte textos a matriz numérica (500 documentos x N palabras)
matriz_tfidf_turismo = vectorizer_turismo.fit_transform(textos_turismo)

# Imprime el tamaño de la matriz resultante
print(f"Corpus turismo: {matriz_tfidf_turismo.shape}")

Corpus turismo: (500, 116)


### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [2]:
data_dir = "../data"  # Define la carpeta donde están los libros
gutenberg_texts = []  # Lista para guardar el contenido de cada libro
gutenberg_filenames = []  # Lista para guardar los nombres de archivos

# Recorre todos los archivos en la carpeta
for filename in os.listdir(data_dir):
    # Si es un archivo .txt y aún no hemos cargado 1000 libros
    if filename.endswith('.txt') and len(gutenberg_texts) < 1000:
        try:
            # Abre y lee el contenido del archivo
            with open(os.path.join(data_dir, filename), 'r', encoding='utf-8') as f:
                text = f.read()
                # Si el archivo tiene más de 1000 caracteres (no es muy pequeño)
                if len(text) > 1000:
                    gutenberg_texts.append(text)  # Agrega el texto a la lista
                    gutenberg_filenames.append(filename)  # Agrega el nombre del archivo
        except:  # Si hay error al leer el archivo, lo ignora
            pass

print(f"Corpus Gutenberg cargado: {len(gutenberg_texts)} libros")

Corpus Gutenberg cargado: 1000 libros


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [3]:
# Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`
# Crea un vectorizador TF-IDF para el corpus Gutenberg
vectorizer_gutenberg = TfidfVectorizer()

# Entrena el vectorizador y convierte los 1000 libros a matriz numérica
matriz_tfidf_gutenberg = vectorizer_gutenberg.fit_transform(gutenberg_texts)

# Imprime el tamaño de la matriz (1000 documentos x N palabras únicas)
print(f"Corpus Gutenberg: {matriz_tfidf_gutenberg.shape}")

Corpus Gutenberg: (1000, 930880)


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [4]:
def buscar(query, vectorizer, matriz_tfidf, filenames, top_k=5):
    # Convierte la consulta en vector TF-IDF usando el mismo vectorizador
    query_tfidf = vectorizer.transform([query])
    
    # Calcula similitud coseno entre la consulta y todos los documentos
    similarities = cosine_similarity(query_tfidf, matriz_tfidf)[0]
    
    # Obtiene los índices de los top_k documentos más similares (ordenados de mayor a menor)
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    # Retorna lista de tuplas (nombre_archivo, puntuación) solo si la similitud > 0
    return [(filenames[i], similarities[i]) for i in top_indices if similarities[i] > 0]

# Mensaje confirmando que la función está lista
print("Función buscar() lista")

Función buscar() lista


### Paso 5: Ejemplos de búsqueda

In [5]:
queries = ["love nature journey", "mystery crime", "adventure discovery", "science invention"]  # Lista de consultas para probar

# Itera sobre cada consulta
for q in queries:
    print(f"\nBúsqueda: '{q}'")  # Imprime la consulta que se está buscando
    
    # Llama a la función buscar() en el corpus Gutenberg con los parámetros especificados
    resultados = buscar(q, vectorizer_gutenberg, matriz_tfidf_gutenberg, gutenberg_filenames, top_k=3)
    
    # Itera sobre los resultados (nombre del libro y puntuación)
    for libro, score in resultados:
        # Imprime el nombre del libro y la puntuación de similitud con 4 decimales
        print(f"  {libro}: {score:.4f}")


Búsqueda: 'love nature journey'
  Helen of Troy, and Other Poems.txt: 0.0568
  Ethics — Part 1.txt: 0.0547
  Love Songs.txt: 0.0520

Búsqueda: 'mystery crime'
  Criminal Sociology.txt: 0.0495
  A Book of Remarkable Criminals.txt: 0.0207
  The Hacker Crackdown_ Law and Disorder on the Electronic Frontier.txt: 0.0151

Búsqueda: 'adventure discovery'
  Young Adventure_ A Book of Poems.txt: 0.0078
  John Barleycorn.txt: 0.0061
  French Cave Paintings.txt: 0.0059

Búsqueda: 'science invention'
  Industrial Biography_ Iron Workers and Tool Makers.txt: 0.0208
  History of the Warfare of Science with Theology in Christendom.txt: 0.0193
  NREN for All_ Insurmountable Opportunity.txt: 0.0182
